# Geospatial Data Analysis and Visualization
## Class 09 — Vector Data Manipulation

**Keynote Speaker:** Md Wabidur Rahman, Researcher, Research and Integrated Thought (RIT)
**Class 09 · 16 October 2026**

---

### What this notebook covers

| Part | Topic | Main tool |
|------|------|---------|
| 1 | **CRS** — what a coordinate system is, and why | `pyproj`, `.crs` |
| 2 | **The area disaster** — degrees versus metres | `.to_crs()` |
| 3 | **Geometric attributes** — area, length, centroid | `.area` `.centroid` |
| 4 | **Spatial Join** — matching the ERA5 grid to the districts | `gpd.sjoin()` |
| 5 | **Dissolve, Overlay, Clip, Buffer** | `.dissolve()` `.overlay()` |
| 6 | **Geocoding** — address to coordinate and back | `geopy` |
| 7 | **Building an analysis-ready dataset** | `.to_parquet()` |

---

### Today's goal

In Class 08 (Part-1) we saw that the ERA5 grid is **larger** than Bangladesh — it also covers India,
Myanmar and the Bay of Bengal. Today we learn how to —

1. measure in the correct units (square kilometres), and
2. **attach every ERA5 grid point to its district**

That second job is the backbone of the whole study — it is what will let us say
"what is the mean temperature in Rangamati district" or "which district is warming fastest".

In [ ]:
# ════════════════════════════════════════════════════════════════════
#  SETUP — locating and preparing the data  (identical in every notebook)
# ════════════════════════════════════════════════════════════════════
# This cell finds your four datasets by itself:
#   * monthly_mean_data_*.zip / data_0.nc   (ERA5 monthly)
#   * hourly_mean_data_*.zip  / data_0.nc   (ERA5 hourly)
#   * BGD_adm.zip                           (GADM shapefile)
#   * bgd_adm_bbs_*_shp.zip                 (BBS/HDX shapefile)
# If it finds the zips it extracts them into ./data/. If they are already extracted it uses them as they are.

from pathlib import Path
import zipfile, sys, os

DATA = Path.cwd() / "data"
OUT  = Path.cwd() / "outputs"
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)

# Where to look
SEARCH_ROOTS = [Path.cwd(), Path.cwd()/"data", Path.cwd().parent,
                Path.home()/"Downloads", Path.home()/"Desktop", Path.home()]

def _find(patterns, roots=SEARCH_ROOTS, maxdepth=3):
    """Return the first file that matches the pattern."""
    for root in roots:
        if not root.exists():
            continue
        for pat in patterns:
            try:
                hits = sorted(root.glob(pat)) + sorted(root.glob("*/" + pat))
                if maxdepth >= 3:
                    hits += sorted(root.glob("*/*/" + pat))
                for h in hits:
                    if h.exists():
                        return h
            except Exception:
                pass
    return None

def _ensure(zip_patterns, member_pattern, dest):
    """If member_pattern is not in dest, find the zip and extract it there."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    got = sorted(dest.rglob(member_pattern))
    if got:
        return got[0]
    z = _find(zip_patterns)
    if z is not None:
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        got = sorted(dest.rglob(member_pattern))
        if got:
            print(f"  extracted {z.name}  ->  {dest.name}/")
            return got[0]
    return None

print("Looking for the data ...")
NC_MONTHLY = _ensure(["*monthly_mean*zip"], "*.nc", DATA/"nc_monthly")
NC_HOURLY  = _ensure(["*hourly_mean*zip"],  "*.nc", DATA/"nc_hourly")
SHP_GADM   = _ensure(["*BGD_adm*.zip"], "BGD_adm2.shp", DATA/"shp_adm")
SHP_BBS    = _ensure(["*bgd_adm_bbs*shp*.zip", "*bgd_adm_bbs*.zip"],
                     "bgd_admbnda_adm2*.shp", DATA/"shp_bbs")

# GADM is optional -- only the GADM versus BBS comparison in Class 08 needs
# it (see the README).
status = {"ERA5 monthly (.nc)": NC_MONTHLY, "ERA5 hourly (.nc)": NC_HOURLY,
          "BBS/HDX shapefile": SHP_BBS}
print()
ok = True
for k, v in status.items():
    print(f"  {'✓' if v else '✗'}  {k:24s} {v if v else 'not found'}")
    ok = ok and (v is not None)
print(f"  {'✓' if SHP_GADM else '·'}  {'GADM (optional)':24s} "
      f"{SHP_GADM if SHP_GADM else 'absent -- only the Class 08 comparison is skipped'}")
print()
if ok:
    print("All required data ready")
else:
    print("Some required files were not found — put the zip files next to this notebook and run again.")

In [ ]:
# -- Libraries and shared styling (identical in every notebook) ------------
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
import geopandas as gpd, xarray as xr

# Publication-ready defaults so the figures can go straight into a journal
plt.rcParams.update({
    "figure.dpi": 110,          # for viewing on screen
    "savefig.dpi": 300,         # for saving to a file (journals usually want >=300)
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 10, "axes.grid": True,
    "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "figure.autolayout": False,
})

# Brand colours — the same in every notebook
C_WARM, C_COOL, C_ACC, C_INK = "#B4562F", "#3D7EA6", "#E0A458", "#1B3A5C"

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("geopandas   ", gpd.__version__)
print("xarray      ", xr.__version__)
print("matplotlib  ", matplotlib.__version__)

In [ ]:
# today's data — BBS boundaries and the ERA5 grid
BBSDIR = SHP_BBS.parent
country  = gpd.read_file(sorted(BBSDIR.glob("bgd_admbnda_adm0_*.shp"))[0])
division = gpd.read_file(sorted(BBSDIR.glob("bgd_admbnda_adm1_*.shp"))[0])
district = gpd.read_file(sorted(BBSDIR.glob("bgd_admbnda_adm2_*.shp"))[0])
upazila  = gpd.read_file(sorted(BBSDIR.glob("bgd_admbnda_adm3_*.shp"))[0])

district = district[["ADM2_EN", "ADM2_PCODE", "ADM1_EN", "geometry"]].copy()
division = division[["ADM1_EN", "ADM1_PCODE", "geometry"]].copy()
upazila  = upazila[["ADM3_EN", "ADM2_EN", "ADM1_EN", "geometry"]].copy()

ds = xr.open_dataset(NC_MONTHLY).squeeze(drop=True)
ds["t_c"] = ds["t"] - 273.15

print(f"country   : {len(country)}")
print(f"divisions : {len(division)}")
print(f"districts : {len(district)}")
print(f"upazilas  : {len(upazila)}")
print(f"ERA5 grid : {ds.sizes['latitude']} x {ds.sizes['longitude']} = "
      f"{ds.sizes['latitude']*ds.sizes['longitude']} cells, {ds.sizes['valid_time']} months")

---
# Part 1 — Coordinate Reference System (CRS)

## 1.1 What is the problem?

The Earth is a **curved, near-spherical** body. Paper and a screen are **flat**. Bringing a curved
surface onto a flat one **must distort something** — this is mathematically unavoidable.

A **CRS** states what place on Earth the numbers (such as `90.41, 23.81`) refer to.
There are two kinds —

| | **Geographic CRS** | **Projected CRS** |
|---|---|---|
| Unit | degrees (°) | metres (m) |
| Basis | angles on a sphere | flattened onto a plane |
| Example | **EPSG:4326** (WGS 84) | **EPSG:32646** (UTM 46N) |
| Used for | stating a location, GPS, exchanging data | **measurement** — length, area |
| Drawing maps | fine | fine |
| Measuring area | **wrong** | **correct** |

> **What is EPSG?** *European Petroleum Survey Group* — the international register that gives every
> CRS a number. Write `EPSG:4326` and the whole world knows you mean WGS 84 (the system GPS uses).

### Which CRS for Bangladesh?

| CRS | When to use it |
|-----|-------------------|
| **EPSG:4326** | storing and exchanging data, web maps, GPS coordinates |
| **EPSG:32646** (UTM 46N) | measurement over **most** of Bangladesh (east of 90°E) |
| **EPSG:32645** (UTM 45N) | western Bangladesh (west of 90°E) |
| **EPSG:3106** (BTM) | Bangladesh's **official** national system — the whole country at once |

> **The UTM rule:** the Earth is divided into 60 zones each 6° wide. Bangladesh falls across zones 45
> and 46. To measure the whole country at once, **EPSG:3106 (BTM)** is the best choice —
> it was built for Bangladesh.

In [ ]:
from pyproj import CRS

print("The CRS of our data:")
print(" ", district.crs)
print()

crs_list = {
    "EPSG:4326  WGS 84 (GPS)"            : 4326,
    "EPSG:32645 UTM zone 45N"            : 32645,
    "EPSG:32646 UTM zone 46N"            : 32646,
    "EPSG:3106  Gulshan 303 / TM 90 NE"  : 3106,
}
rows = []
for label, code_ in crs_list.items():
    c = CRS.from_epsg(code_)
    rows.append({
        "CRS": label,
        "kind": "projected" if c.is_projected else "geographic",
        "unit": c.axis_info[0].unit_name,
        "area of use": (c.area_of_use.name[:42] if c.area_of_use else "-"),
    })
pd.DataFrame(rows).set_index("CRS")

---
# Part 2 — The area disaster (the most important lesson)

Now let us see for ourselves **what happens when you measure in the wrong CRS**.

In [ ]:
# WRONG — taking area directly in EPSG:4326 (degrees)
wrong = district.geometry.area.sum()
print("WRONG: area in EPSG:4326 =", round(wrong, 4), " 'square degrees'")
print("   This number has no physical meaning!\n")

# CORRECT — convert to a projected CRS first, then take the area
for epsg, name in [(32645, "UTM 45N"), (32646, "UTM 46N"), (3106, "BTM (Bangladesh)")]:
    km2 = district.to_crs(epsg).geometry.area.sum() / 1e6
    print(f"✅ EPSG:{epsg} {name:20s} = {km2:10,.0f} km²")

print("\n-> the three projections agree to within ~0.3% of each other — acceptable.")
print("-> but the degree figure belongs to another world entirely — it cannot be compared.")

### Why can you not measure in degrees?

One degree of **latitude** is always about 111 km. But one degree of **longitude** —

| Latitude | 1° of longitude = how many km |
|---------|--------------------------|
| 0° (the equator) | 111.3 |
| 23.8° (Dhaka) | 101.9 |
| 60° | 55.8 |
| 90° (the pole) | 0 |

The formula: `distance = 111.32 × cos(latitude)` km

So a "square degree" is a **varying unit** — one square degree means a different amount of ground in
Dhaka than in Rangpur. That is why adding them up is meaningless.

In [ ]:
# let us verify the formula above
lats = np.array([0, 23.8, 45, 60, 90])
km_per_deg_lon = 111.32 * np.cos(np.deg2rad(lats))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
la = np.linspace(0, 90, 200)
ax[0].plot(la, 111.32*np.cos(np.deg2rad(la)), color=C_WARM, lw=2, label="1° longitude")
ax[0].axhline(111.32, color=C_COOL, ls="--", lw=1.5, label="1° latitude (constant)")
ax[0].axvline(23.8, color="gray", ls=":", lw=1.2)
ax[0].text(24.6, 95, "Bangladesh\n(~23.8°N)", fontsize=8)
ax[0].set_xlabel("Latitude (°)"); ax[0].set_ylabel("Ground distance (km)")
ax[0].set_title("Why 1° is not a fixed distance"); ax[0].legend()

# how the same division looks when drawn in two CRSs
dhaka_div = district[district["ADM1_EN"] == "Dhaka"]
dhaka_div.plot(ax=ax[1], ec="k", lw=0.4, color="#cfe3f7")
ax[1].set_title("Dhaka division in EPSG:4326 (degrees)")
ax[1].set_xlabel("Longitude (°E)"); ax[1].set_ylabel("Latitude (°N)")
plt.tight_layout(); plt.show()

for l, k in zip(lats, km_per_deg_lon):
    print(f"  {l:5.1f} deg  ->  1 deg of longitude = {k:6.2f} km")

In [ ]:
# the correct area of each district (in BTM)
dist_m = district.to_crs(3106).copy()
dist_m["area_km2"] = dist_m.geometry.area / 1e6
dist_m["perimeter_km"] = dist_m.geometry.length / 1e3

print("The five largest districts:")
print(dist_m.nlargest(5, "area_km2")[["ADM2_EN", "ADM1_EN", "area_km2"]]
      .round(1).to_string(index=False))
print("\nThe five smallest districts:")
print(dist_m.nsmallest(5, "area_km2")[["ADM2_EN", "ADM1_EN", "area_km2"]]
      .round(1).to_string(index=False))
print(f"\nTotal area: {dist_m['area_km2'].sum():,.0f} km2")

> **⚠ A caution for research:** the total area above (~140,000 km²) is smaller than the
> commonly cited figure of 147,570 km² for Bangladesh. The reason is that the result depends on how
> the boundary dataset treats large rivers and estuaries, and the polygons are somewhat generalised.
>
> **The lesson:** the area you compute is **dataset-dependent**. In a journal you must write which
> boundary file and which CRS you used. Never claim your own figure is the "official" one.

---
# Part 3 — Geometric attributes

| Attribute | What it gives | Needs a CRS? |
|-----------|---------|------------|
| `.area` | area | yes, projected |
| `.length` | perimeter / length | yes, projected |
| `.centroid` | the geometric centre | yes, projected |
| `.representative_point()` | a point **guaranteed to be inside** | yes, projected |
| `.bounds` | the bounding box | any |
| `.total_bounds` | the bounding box of the whole layer | any |

In [ ]:
# centroid — computed in a projected CRS, then brought back to 4326 (for placing on a map)
cent_m = dist_m.copy()
cent_m["centroid"] = cent_m.geometry.centroid
cent_ll = gpd.GeoSeries(cent_m["centroid"], crs=3106).to_crs(4326)
dist_m["cent_lon"] = cent_ll.x.values
dist_m["cent_lat"] = cent_ll.y.values

print("Centroids (the first five):")
print(dist_m[["ADM2_EN", "cent_lon", "cent_lat", "area_km2"]].head()
      .round(3).to_string(index=False))

print("\nBounding box of the whole of Bangladesh (EPSG:4326):")
minx, miny, maxx, maxy = country.total_bounds
print(f"  longitude {minx:.3f}E -> {maxx:.3f}E")
print(f"  latitude  {miny:.3f}N -> {maxy:.3f}N")

> **⚠ `centroid` versus `representative_point()`:** the geometric centre of a **curved or hollow**
> polygon (a "C" shape, say) can fall **outside** the polygon! To place a label, or to get "a point
> inside this area", `.representative_point()` is the safe choice — it is always inside.

In [ ]:
# which districts have a centroid that falls outside themselves?
chk = district.to_crs(3106)
inside = chk.geometry.centroid.within(chk.geometry)
outside = district.loc[~inside.values, "ADM2_EN"].tolist()
print(f"Districts whose centroid falls outside their own boundary: {len(outside)}")
if outside:
    print(" ", ", ".join(outside))
    print("-> these are usually island districts or very concave shapes.")
print("\nFix: use representative_point() for labels.")

rep = chk.geometry.representative_point()
print("Is representative_point always inside?", bool(rep.within(chk.geometry).all()))

In [ ]:
# a map of district area — our first choropleth
fig, ax = plt.subplots(1, 2, figsize=(13, 6.4))

dist_m.to_crs(4326).plot(column="area_km2", ax=ax[0], cmap="YlGnBu",
                         legend=True, edgecolor="white", linewidth=0.3,
                         legend_kwds={"label": "Area (km²)", "shrink": 0.75})
ax[0].set_title("District area (BTM, EPSG:3106)")

# with the centroids
district.boundary.plot(ax=ax[1], color="gray", linewidth=0.3)
ax[1].scatter(dist_m["cent_lon"], dist_m["cent_lat"],
              s=dist_m["area_km2"]/40, c=C_WARM, alpha=0.65,
              edgecolor="k", linewidth=0.4)
ax[1].set_title("District centroids (size ∝ area)")

for a in ax:
    a.set_xlabel("Longitude (°E)"); a.set_ylabel("Latitude (°N)")
plt.tight_layout(); plt.show()

---
# Part 4 — Spatial Join: matching the ERA5 grid to the districts

## 4.1 The idea

An ordinary `pandas.merge` joins two tables on **a shared column** (an id, say).
A **spatial join** joins them on **location** —
"which polygon is this point inside?"

This is today's most important job. Our ERA5 grid has 841 points, and we want to know
**which district** each one falls in.

| predicate | Meaning |
|-----------|------|
| `within` | the left geometry is **inside** the right one |
| `contains` | the left one **contains** the right one |
| `intersects` | the two **touch or cross** |
| `touches` | they touch only **at the boundary** |

In [ ]:
# Step 1 — turning the ERA5 grid into a GeoDataFrame of points
lat = ds.latitude.values
lon = ds.longitude.values
LON, LAT = np.meshgrid(lon, lat)

grid = gpd.GeoDataFrame(
    {"lat": LAT.ravel(), "lon": LON.ravel(),
     "t_mean": ds.t_c.mean("valid_time").values.ravel(),
     "rh_mean": ds.r.mean("valid_time").values.ravel()},
    geometry=gpd.points_from_xy(LON.ravel(), LAT.ravel()),
    crs="EPSG:4326")

print(f"ERA5 grid points: {len(grid)}  ({len(lat)} x {len(lon)})")
grid.head(3)

In [ ]:
# Step 2 — the spatial join: which point is in which district?
joined = gpd.sjoin(grid, district[["ADM2_EN", "ADM1_EN", "geometry"]],
                   how="inner", predicate="within")

print(f"total grid points            : {len(grid)}")
print(f"falling inside Bangladesh    : {len(joined)}  "
      f"({100*len(joined)/len(grid):.1f}%)")
print(f"outside (India/Myanmar/sea)  : {len(grid)-len(joined)}")
print(f"\nDistricts with at least one point: {joined['ADM2_EN'].nunique()} / {len(district)}")

joined.head(3)[["lat", "lon", "t_mean", "ADM2_EN", "ADM1_EN"]]

### ⚠ A real problem — some districts got no point at all

An ERA5 cell is about 28 km across, but some districts are **smaller** than that
(Narayanganj is 704 km² ≈ 26 × 26 km). So no grid point may fall inside such a district.

This is the **coarse grid versus small polygon** problem — very common in real research.
There are three solutions, and all three are shown below.

In [ ]:
# which districts received no point?
counts = joined.groupby("ADM2_EN").size().rename("n_points")
gap = district[~district["ADM2_EN"].isin(counts.index)]["ADM2_EN"].tolist()
print(f"Districts with zero points ({len(gap)}): {', '.join(gap) if gap else 'none'}")
print()
print("Distribution of the point counts:")
print(counts.describe().round(2).to_string())
print(f"\nMost points: {counts.idxmax()} ({counts.max()})")

In [ ]:
# -- Solution 1: sjoin_nearest — take the nearest point
near = gpd.sjoin_nearest(
    district[["ADM2_EN", "geometry"]].to_crs(3106),
    grid.to_crs(3106), how="left", distance_col="dist_m")
sol1 = near.groupby("ADM2_EN").agg(t_mean=("t_mean", "mean"),
                                   rh_mean=("rh_mean", "mean")).reset_index()
print(f"Solution 1 (sjoin_nearest)   : a value for {sol1['t_mean'].notna().sum()} / {len(district)} districts")

# -- Solution 2: the grid cell nearest the polygon centre (xarray)
cent = district.to_crs(3106).geometry.centroid
cll = gpd.GeoSeries(cent, crs=3106).to_crs(4326)
vals = [float(ds.t_c.mean("valid_time").sel(latitude=y, longitude=x, method="nearest"))
        for x, y in zip(cll.x, cll.y)]
sol2 = pd.DataFrame({"ADM2_EN": district["ADM2_EN"].values, "t_mean": vals})
print(f"Solution 2 (centroid nearest): a value for {sol2['t_mean'].notna().sum()} / {len(district)} districts")

# -- Solution 3: an all_touched raster clip (detailed in Notebook 3)
print(f"Solution 3 (all_touched clip): in Notebook 3 — the most accurate, area-based")

cmp = sol1.merge(sol2, on="ADM2_EN", suffixes=("_nearest", "_centroid"))
cmp["difference"] = (cmp["t_mean_nearest"] - cmp["t_mean_centroid"]).abs()
print(f"\nMean difference between the two methods: {cmp['difference'].mean():.3f} C "
      f"(maximum {cmp['difference'].max():.3f} C)")
print("-> a small difference, but you must state in Methods which one you used.")

In [ ]:
# the inside and outside points on a map
fig, ax = plt.subplots(figsize=(7.5, 7))
outside = grid[~grid.index.isin(joined.index)]

country.plot(ax=ax, color="#eef4fa", edgecolor="k", linewidth=1.2, zorder=1)
district.boundary.plot(ax=ax, color="gray", linewidth=0.25, zorder=2)
ax.scatter(outside["lon"], outside["lat"], s=11, c="lightgray",
           label=f"Outside Bangladesh (n={len(outside)})", zorder=3)
ax.scatter(joined["lon"], joined["lat"], s=16, c=C_WARM,
           label=f"Inside Bangladesh (n={len(joined)})", zorder=4,
           edgecolor="k", linewidth=0.2)

ax.set_title("ERA5 grid points vs Bangladesh districts\n"
             "spatial join with predicate='within'")
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# mean temperature by district — our first real analytical result
zonal = joined.groupby(["ADM1_EN", "ADM2_EN"]).agg(
    t_mean=("t_mean", "mean"), rh_mean=("rh_mean", "mean"),
    n_points=("t_mean", "size")).reset_index()

print("The five warmest districts (25-year mean):")
print(zonal.nlargest(5, "t_mean")[["ADM2_EN", "ADM1_EN", "t_mean", "n_points"]]
      .round(2).to_string(index=False))
print("\nThe five coolest districts:")
print(zonal.nsmallest(5, "t_mean")[["ADM2_EN", "ADM1_EN", "t_mean", "n_points"]]
      .round(2).to_string(index=False))
print("\nMeans by division:")
print(zonal.groupby("ADM1_EN")[["t_mean", "rh_mean"]].mean()
      .round(2).sort_values("t_mean", ascending=False).to_string())

---
# Part 5 — Dissolve, Overlay, Clip, Buffer

| Operation | What it does | Example |
|-----|---------|---------|
| `dissolve` | merges small units into larger ones | upazila -> district -> division |
| `overlay` | geometric combination of two layers | intersection, union, difference |
| `clip` | cuts one layer to the shape of another | cutting the grid to the country |
| `buffer` | a belt at a fixed distance around something | the area within 5 km of a river |

In [ ]:
# -- Dissolve: upazila -> district -> division
up2dist = upazila.dissolve(by="ADM2_EN").reset_index()
dist2div = district.dissolve(by="ADM1_EN").reset_index()

print(f"upazilas  {len(upazila)}  ->  dissolve  ->  districts {len(up2dist)}")
print(f"districts {len(district)}  ->  dissolve  ->  divisions {len(dist2div)}")

fig, ax = plt.subplots(1, 3, figsize=(14, 4.8))
for a, g, ttl in [(ax[0], upazila, f"Upazilas (n={len(upazila)})"),
                  (ax[1], district, f"Districts (n={len(district)})"),
                  (ax[2], dist2div, f"Divisions (n={len(dist2div)})")]:
    g.plot(ax=a, edgecolor="k", linewidth=0.2, color="#cfe3f7")
    a.set_title(ttl); a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.suptitle("Dissolve — merging small units into larger ones",
             fontweight="bold", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# -- Buffer: the belt 20 km inland from the coast (as in cyclone-risk analysis)
coast_div = ["Barisal", "Chittagong", "Khulna"]
coastal = district[district["ADM1_EN"].isin(coast_div)].to_crs(3106)

# 20 km inward from the national boundary
cshape = country.to_crs(3106)
inner = cshape.buffer(-20000)                   # a negative buffer shrinks inward
coastal_belt = cshape.difference(inner)         # the outer 20 km belt

area_total = cshape.area.sum()/1e6
area_belt  = coastal_belt.area.sum()/1e6
print(f"total area of the country    : {area_total:,.0f} km2")
print(f"the 20 km belt from the edge : {area_belt:,.0f} km2  "
      f"({100*area_belt/area_total:.1f}%)")

fig, ax = plt.subplots(figsize=(6.5, 6.5))
gpd.GeoSeries(inner, crs=3106).to_crs(4326).plot(ax=ax, color="#cfe3f7",
                                                 ec="none", label="Interior")
gpd.GeoSeries(coastal_belt, crs=3106).to_crs(4326).plot(ax=ax, color=C_WARM,
                                                 ec="none", alpha=0.75)
country.boundary.plot(ax=ax, color="k", lw=1.0)
ax.set_title("20 km inward buffer from the national boundary")
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
plt.tight_layout(); plt.show()

In [ ]:
# ── Overlay & Clip
sylhet = division[division["ADM1_EN"] == "Sylhet"]

# clip — cutting the upazila layer to the shape of Sylhet division
syl_up = gpd.clip(upazila, sylhet)
print(f"clip: {len(upazila)} upazilas nationwide  ->  {len(syl_up)} in Sylhet division")

# overlay — the intersection of two layers
box = gpd.GeoDataFrame(geometry=[gpd.GeoSeries(
    gpd.points_from_xy([90.0, 92.5], [23.5, 25.5])).total_bounds.tolist()
    and __import__("shapely").geometry.box(90.0, 23.5, 92.5, 25.5)],
    crs="EPSG:4326")
inter = gpd.overlay(district, box, how="intersection")
print(f"overlay(intersection): {len(inter)} districts intersect the rectangle")

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
syl_up.plot(ax=ax[0], ec="k", lw=0.3, color="#9ec9e8")
sylhet.boundary.plot(ax=ax[0], color=C_WARM, lw=1.6)
ax[0].set_title(f"clip — Sylhet division ({len(syl_up)} upazilas)")

district.boundary.plot(ax=ax[1], color="lightgray", lw=0.3)
inter.plot(ax=ax[1], color=C_ACC, ec="k", lw=0.3, alpha=0.85)
box.boundary.plot(ax=ax[1], color=C_WARM, lw=1.8, ls="--")
ax[1].set_title(f"overlay — intersection with a bounding box")
for a in ax: a.set_xlabel("Longitude (°E)"); a.set_ylabel("Latitude (°N)")
plt.tight_layout(); plt.show()

---
# Part 6 — Building an analysis-ready dataset

Now we join everything into one **analysis-ready** table — Notebooks 3 and 4 will use it.
In journal work this step matters a great deal: from raw data to one clean, documented,
reproducible file.

In [ ]:
# district + area + ERA5 climate (the gaps filled with sjoin_nearest)
base = district.merge(
    dist_m[["ADM2_EN", "area_km2", "perimeter_km", "cent_lon", "cent_lat"]],
    on="ADM2_EN", how="left")

base = base.merge(zonal[["ADM2_EN", "t_mean", "rh_mean", "n_points"]],
                  on="ADM2_EN", how="left")

# filling the zero-point districts from the nearest grid cell
miss = base["t_mean"].isna()
if miss.any():
    print(f"{miss.sum()} districts have no grid point of their own -> filled from the nearest cell")
    for i in base.index[miss]:
        x, y = base.loc[i, "cent_lon"], base.loc[i, "cent_lat"]
        base.loc[i, "t_mean"] = float(ds.t_c.mean("valid_time")
                                      .sel(latitude=y, longitude=x, method="nearest"))
        base.loc[i, "rh_mean"] = float(ds.r.mean("valid_time")
                                       .sel(latitude=y, longitude=x, method="nearest"))
        base.loc[i, "n_points"] = 0

base["n_points"] = base["n_points"].fillna(0).astype(int)
print(f"\nAny missing values left? {int(base[['t_mean','rh_mean','area_km2']].isna().sum().sum())}")
print(f"total districts: {len(base)}")
base.head(3)[["ADM2_EN", "ADM1_EN", "area_km2", "t_mean", "rh_mean", "n_points"]].round(2)

In [ ]:
# the final figure — mean temperature and humidity by district
fig, ax = plt.subplots(1, 2, figsize=(13.5, 6.2))

base.plot(column="t_mean", ax=ax[0], cmap="RdYlBu_r", legend=True,
          edgecolor="white", linewidth=0.3,
          legend_kwds={"label": "Mean temperature (°C)", "shrink": 0.75})
ax[0].set_title("(a) District mean air temperature, 2000–2024")

base.plot(column="rh_mean", ax=ax[1], cmap="YlGnBu", legend=True,
          edgecolor="white", linewidth=0.3,
          legend_kwds={"label": "Mean relative humidity (%)", "shrink": 0.75})
ax[1].set_title("(b) District mean relative humidity, 2000–2024")

for a in ax:
    division.boundary.plot(ax=a, color="k", linewidth=0.6)
    a.set_xlabel("Longitude (°E)"); a.set_ylabel("Latitude (°N)")
plt.tight_layout()
plt.savefig(OUT/"fig_nb2_district_climate.png", dpi=300, bbox_inches="tight")
plt.show()
print("Figure saved: outputs/fig_nb2_district_climate.png (300 dpi)")

---
# Home Work — Class 09

**1.** Convert the `division` layer to **EPSG:3106** and find the area (km²) of each division.
Which division is the largest, and which the smallest?

**2.** Compute the district areas in **EPSG:32645**, **EPSG:32646** and **EPSG:3106** and show them
side by side in one table. What is the percentage difference?

**3.** Do a spatial join between the `upazila` layer and the ERA5 grid points. How many
upazilas got no point at all? (How many times worse is the problem than at district level?)

**4.** Build a **50 km** radius buffer around the city of Dhaka (in EPSG:3106), and find which
districts `intersects` that circle.

**5.** From the `base` table, draw a scatter plot with `area_km2` on the x-axis and
`t_mean` on the y-axis, coloured by division. Are the larger districts warmer?

---
# Class 09 — Summary

| What we learned | Key command |
|-----------|-------------|
| inspecting the CRS | `gdf.crs` · `CRS.from_epsg(3106)` |
| changing the CRS | `gdf.to_crs(3106)` |
| the correct area | `gdf.to_crs(3106).area / 1e6` -> km² |
| the centre point | `.centroid` · `.representative_point()` |
| grid -> points | `gpd.points_from_xy()` |
| the spatial join | `gpd.sjoin(..., predicate="within")` |
| filling the gaps | `gpd.sjoin_nearest()` |
| merging units | `.dissolve(by=...)` |
| cutting / intersecting | `gpd.clip()` · `gpd.overlay()` |
| a belt | `.buffer(distance_in_metres)` |
| address -> coordinate | `Nominatim().geocode()` |

---

### The most common mistakes

1. **Taking `.area` or `.length` in EPSG:4326** — the result is in square degrees, which is meaningless. Use `.to_crs()` first.
2. **The wrong unit in a buffer** — in a projected CRS `.buffer(5000)` means 5 km; but in 4326 it means 5000 **degrees**!
3. **Joining two layers in different CRSs** — bring both to the same CRS first, or the result will be wrong or empty.
4. **Not checking for gaps after an sjoin** — a coarse grid may leave small polygons with no point.
5. **A centroid falling outside** — for labels, `representative_point()` is safer.

### What comes next

**Class 10 — Raster Data & Remote Sensing**: using the NetCDF as a true raster,
cutting it to the shape of Bangladesh, zonal statistics, GeoTIFF/COG, and
using satellite imagery **without downloading it** (STAC).

---

**Md Wabidur Rahman** — Researcher, Research and Integrated Thought (RIT)
*Geospatial Data Analysis and Visualization · Class 09 — Vector Data Manipulation*